# BIG DATA ANALYSIS WITH SCALA AND SPARK LAB (ENSP359)

**Name:** Aditya Shibu  
**Roll Number:** 2401201047  
**Course:** BCA (AI & DS)  
**University:** K.R. Mangalam University  
**School:** School of Engineering & Technology  
**Subject:** BIG DATA ANALYSIS WITH SCALA AND SPARK LAB (ENSP359)  
**Submitted To:** Mr. NIKHIL SHARMA

---

## Setup Spark Environment
Configured for Java 17 compatibility on Arch Linux.

In [1]:
import os
import sys
import subprocess

# Force Java 17 for Spark stability
if os.path.exists("/usr/lib/jvm/java-17-openjdk"):
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk"

os.environ["PYSPARK_SUBMIT_ARGS"] = '--driver-java-options "--add-opens=java.base/sun.security.action=ALL-UNNAMED" pyspark-shell'

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import *
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "pyspark", "--break-system-packages"])
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import *

spark = SparkSession.builder.master("local[*]").appName("BigDataLab") \
    .config("spark.sql.parquet.compression.codec", "zstd") \
    .getOrCreate()
sc = spark.sparkContext
print("Spark Session Active.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 00:19:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/23 00:19:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Session Active.


### Ex 1: Basics (Variables, Types, Control Flow)
**Objective:** Arithmetic, conditions, loops, and functions.

In [2]:
a, b = 10, 20.5
print(f"Sum: {a + b}")

def check_val(x):
    if x > 15: return "Large"
    else: return "Small"

for i in range(1, 4):
    print(f"Val {i}: {check_val(i * 10)}")

Sum: 30.5
Val 1: Small
Val 2: Large
Val 3: Large


### Ex 2: Functions for Collections
**Objective:** Lists, Vectors, and Maps.

In [3]:
my_list = [1, 2, 3]
my_map = {"name": "Aditya", "roll": 47}
print(f"List: {my_list}")
print(f"Map access: {my_map['name']}")

List: [1, 2, 3]
Map access: Aditya


### Ex 3: Transformations
**Objective:** map, filter, flatMap.

In [4]:
data = [1, 2, 3, 4, 5]
rdd = sc.parallelize(data)

print(f"Map: {rdd.map(lambda x: x*2).collect()}")
print(f"Filter: {rdd.filter(lambda x: x > 3).collect()}")

Map: [2, 4, 6, 8, 10]
Filter: [4, 5]


### Ex 4: Word Count (RDD)
**Objective:** Map-reduce pattern with reduceByKey.

In [5]:
text = sc.parallelize(["spark spark python", "big data"])
counts = text.flatMap(lambda x: x.split(" ")) \
             .map(lambda x: (x, 1)) \
             .reduceByKey(lambda a, b: a + b)
print(counts.collect())

[('spark', 2), ('big', 1), ('python', 1), ('data', 1)]


### Ex 5: RDD Actions & Aggregations
**Objective:** collect, count, aggregate.

In [6]:
rdd = sc.parallelize(range(1, 11))
print(f"Count: {rdd.count()}")
print(f"Sum: {rdd.sum()}")

Count: 10
Sum: 55


### Ex 6: Multiple RDDs
**Objective:** union, intersection, subtract.

In [7]:
r1 = sc.parallelize([1, 2, 3])
r2 = sc.parallelize([3, 4, 5])
print(f"Union: {r1.union(r2).collect()}")
print(f"Intersection: {r1.intersection(r2).collect()}")

Union: [1, 2, 3, 3, 4, 5]
Intersection: [3]


### Ex 7: DataFrames basics
**Objective:** Load and select/filter.

In [8]:
df = spark.createDataFrame([("A", 1), ("B", 2)], ["name", "id"])
df.select("name").filter(df.id > 1).show()

+----+
|name|
+----+
|   B|
+----+



### Ex 8: Nested JSON & Explode
**Objective:** Flatten nested data.

In [9]:
data = [("Alice", ["tag1", "tag2"])]
df_json = spark.createDataFrame(data, ["name", "tags"])
df_json.withColumn("tag", explode(col("tags"))).show()

+-----+------------+----+
| name|        tags| tag|
+-----+------------+----+
|Alice|[tag1, tag2]|tag1|
|Alice|[tag1, tag2]|tag2|
+-----+------------+----+



### Ex 9: Missing Data
**Objective:** fillna, dropna.

In [10]:
df_miss = spark.createDataFrame([("A", None), (None, 10)], ["name", "val"])
df_miss.na.fill(0).show()

+----+---+
|name|val|
+----+---+
|   A|  0|
|NULL| 10|
+----+---+



### Ex 10: DataFrame Aggregations
**Objective:** groupBy, sum, avg.

In [11]:
df.groupBy("name").count().show()

+----+-----+
|name|count|
+----+-----+
|   A|    1|
|   B|    1|
+----+-----+



### Ex 11: Spark SQL Views
**Objective:** SQL queries.

In [12]:
df.createOrReplaceTempView("mytable")
spark.sql("SELECT * FROM mytable WHERE id = 2").show()

+----+---+
|name| id|
+----+---+
|   B|  2|
+----+---+



### Ex 12: Joins
**Objective:** Inner/Outer joins.

In [13]:
d1 = spark.createDataFrame([(1, "A")], ["id", "name"])
d2 = spark.createDataFrame([(1, "Sales")], ["id", "dept"])
d1.join(d2, "id").show()

+---+----+-----+
| id|name| dept|
+---+----+-----+
|  1|   A|Sales|
+---+----+-----+



### Ex 13: SQL Column Functions
**Objective:** Apply column transformations.

In [14]:
df.withColumn("id_plus_10", col("id") + 10).show()

+----+---+----------+
|name| id|id_plus_10|
+----+---+----------+
|   A|  1|        11|
|   B|  2|        12|
+----+---+----------+



### Ex 14: File Formats
**Objective:** Parquet and CSV read/write.

In [15]:
df.write.mode("overwrite").parquet("test.parquet")
spark.read.parquet("test.parquet").show()

+----+---+
|name| id|
+----+---+
|   B|  2|
|   A|  1|
+----+---+



### Ex 15: Housing Project Pipeline
**Objective:** End-to-end cleaning and analysis.

In [16]:
h_data = [("House1", 5000, 2), ("House2", 3000, 1)]
h_df = spark.createDataFrame(h_data, ["id", "price", "rooms"])
h_df.withColumn("price_per_room", col("price")/col("rooms")) \
    .groupBy("rooms").avg("price").show()

+-----+----------+
|rooms|avg(price)|
+-----+----------+
|    2|    5000.0|
|    1|    3000.0|
+-----+----------+

